# Reranking in RAG

> **Turning a broad retrieval candidate set into high-quality evidence.**

Our retrieval pipeline has evolved from simple vector search into a system that can combine dense and lexical retrieval.

That gives us a useful candidate set.

But a candidate set is not the same thing as a final ranking.

Suppose retrieval gives us 20 chunks:

```text
Query
  ↓
Retrieval
  ↓
20 candidates
```

Some may be highly relevant.

Some may be loosely related.

Some may contain the right words but answer a different question.

We therefore need another stage that examines the **query and each candidate together** and produces a better relevance ranking.

That stage is called **reranking**.

## What We'll Learn

By the end of this tutorial, you'll understand:

- Why retrieval and reranking are separate problems
- What a candidate set is
- Why retrieving more candidates can improve recall
- Why a candidate set still contains noise
- What a reranker does
- Bi-encoders vs cross-encoders
- Why cross-encoders can provide stronger relevance judgments
- How to use a cross-encoder reranker
- How reranking changes the final candidate order
- The latency and candidate-count trade-offs
- How reranking fits into a production RAG architecture
- Why reranking must be evaluated rather than assumed to help

# 1. Retrieval vs Reranking

It is useful to treat retrieval as multiple stages.

### Retrieval

The first stage asks:

> **Which documents could be relevant?**

It should be relatively fast and should aim to avoid missing useful candidates.

### Reranking

The second stage asks:

> **Which of these candidates are actually the most relevant to this query?**

The distinction is:

```text
Retrieval
    ↓
Find candidates

Reranking
    ↓
Order candidates by relevance
```

These stages have different objectives.

# 2. Why Not Just Retrieve the Top 3?

Suppose the correct chunk is ranked at position 7:

```text
1. Related chunk
2. Related chunk
3. Related chunk
4. Related chunk
5. Related chunk
6. Related chunk
7. Correct chunk
```

If we retrieve only three results, the correct chunk never reaches the next stage.

Instead, we can retrieve a larger candidate set:

```text
Large corpus
    ↓
Fast retrieval
    ↓
20 candidates
    ↓
Reranker
    ↓
Top 5
```

This is the central idea behind multi-stage retrieval.

# 3. Recall First, Precision Later

The first retrieval stage can prioritize **recall**.

We want the correct information to enter the candidate set.

The reranking stage can then prioritize **precision**.

We want the strongest candidates near the top.

Conceptually:

```text
Stage 1
───────
High recall
More candidates
Lower computational cost

        ↓

Stage 2
───────
Higher precision
Fewer candidates
More expensive scoring
```

This gives us an important production pattern:

> **Use a cheaper method to find candidates, then a more expensive method to rank those candidates.**

# 4. Why Similarity Search Isn't Enough

Our dense retriever compares:

```text
Query embedding
      ↓
Document embedding
```

This is efficient because the query and documents can be represented independently.

But there is a limitation.

The embedding model compresses each piece of text into a vector.

A reranker can instead examine the query and candidate **together**:

```text
Query ──────┐
            ├──→ Relevance model
Document ───┘
```

This allows the model to make a more direct judgment about the relationship between the query and the candidate.

# 5. Bi-Encoders and Cross-Encoders

The embedding model we've used is a **bi-encoder-style** approach.

The query and document are encoded independently:

```text
Query
  ↓
Query Vector

Document
  ↓
Document Vector

      ↓

Similarity
```

This makes large-scale retrieval practical.

A **cross-encoder** works differently:

```text
Query + Document
       ↓
Cross-Encoder
       ↓
Relevance Score
```

The model sees both pieces of text together.

This can provide a stronger relevance judgment, but it is more computationally expensive.

That leads naturally to:

```text
Bi-encoder
    ↓
Fast candidate retrieval
    ↓
Cross-encoder
    ↓
Precise reranking
```

# 6. Our Candidate Corpus

We'll use a small corpus where several documents are related to refunds, but only some directly answer a particular question.

In [1]:
documents = [
    {
        "id": "refund-deadline",
        "text": "Customers can request a refund within 30 days of purchase."
    },
    {
        "id": "refund-processing",
        "text": "Approved refunds are normally processed within 7 business days."
    },
    {
        "id": "refund-condition",
        "text": "Products must be returned in their original condition to qualify for a refund."
    },
    {
        "id": "refund-method",
        "text": "Refunds are returned to the original payment method used for the purchase."
    },
    {
        "id": "shipping-standard",
        "text": "Standard shipping normally takes between 3 and 5 business days."
    },
    {
        "id": "support-hours",
        "text": "Customer support is available Monday through Friday from 9 AM to 5 PM."
    },
]

# 7. Build a First-Stage Retriever

We'll use a sentence-transformer model to generate embeddings.

This notebook is self-contained, so install the dependency here.

In [ ]:
!pip install -q sentence-transformers

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

texts = [document["text"] for document in documents]

document_embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
def retrieve_candidates(query, top_k=5):
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    scores = document_embeddings @ query_embedding
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "id": documents[index]["id"],
            "text": documents[index]["text"],
            "retrieval_score": float(scores[index]),
        }
        for index in ranked_indices
    ]

Retrieve a candidate set:

In [4]:
query = "What condition must a product meet before I can receive a refund?"

candidates = retrieve_candidates(query, top_k=5)

for rank, candidate in enumerate(candidates, start=1):
    print(f"Rank {rank}")
    print(f"ID: {candidate['id']}")
    print(f"Retrieval score: {candidate['retrieval_score']:.4f}")
    print(candidate["text"])
    print("---")

Rank 1
ID: refund-condition
Retrieval score: 0.8157
Products must be returned in their original condition to qualify for a refund.
---
Rank 2
ID: refund-deadline
Retrieval score: 0.7596
Customers can request a refund within 30 days of purchase.
---
Rank 3
ID: refund-processing
Retrieval score: 0.7153
Approved refunds are normally processed within 7 business days.
---
Rank 4
ID: refund-method
Retrieval score: 0.6922
Refunds are returned to the original payment method used for the purchase.
---
Rank 5
ID: support-hours
Retrieval score: 0.6124
Customer support is available Monday through Friday from 9 AM to 5 PM.
---


The candidate set can contain several refund-related chunks.

That isn't necessarily a failure.

The first stage is trying to find **plausible candidates**.

The next stage can determine which candidates deserve the highest positions.

# 8. Install a Cross-Encoder Reranker

We'll use a compact cross-encoder from the Sentence Transformers ecosystem:

```text
cross-encoder/ms-marco-MiniLM-L-6-v2
```

This model is suitable for demonstrating passage reranking.

Install the dependency if necessary:

In [ ]:
!pip install -q sentence-transformers

In [5]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

# 9. Rerank the Candidates

A cross-encoder receives pairs:

```text
(query, candidate)
```

For example:

```text
(
    "What condition must a product meet before I can receive a refund?",
    "Products must be returned in their original condition..."
)
```

The model produces a relevance score for each pair.

In [6]:
query_candidate_pairs = [
    (query, candidate["text"])
    for candidate in candidates
]

reranker_scores = reranker.predict(
    query_candidate_pairs
)

reranked_candidates = [
    {
        **candidate,
        "reranker_score": float(score),
    }
    for candidate, score in zip(
        candidates,
        reranker_scores
    )
]

reranked_candidates.sort(
    key=lambda item: item["reranker_score"],
    reverse=True
)

Inspect the reranked results:

In [7]:
for rank, candidate in enumerate(
    reranked_candidates,
    start=1
):
    print(f"Rank {rank}")
    print(f"ID: {candidate['id']}")
    print(f"Retrieval score: {candidate['retrieval_score']:.4f}")
    print(f"Reranker score: {candidate['reranker_score']:.4f}")
    print(candidate["text"])
    print("---")

Rank 1
ID: refund-condition
Retrieval score: 0.8157
Reranker score: 4.2645
Products must be returned in their original condition to qualify for a refund.
---
Rank 2
ID: refund-deadline
Retrieval score: 0.7596
Reranker score: -2.8742
Customers can request a refund within 30 days of purchase.
---
Rank 3
ID: refund-method
Retrieval score: 0.6922
Reranker score: -6.1844
Refunds are returned to the original payment method used for the purchase.
---
Rank 4
ID: refund-processing
Retrieval score: 0.7153
Reranker score: -6.6579
Approved refunds are normally processed within 7 business days.
---
Rank 5
ID: support-hours
Retrieval score: 0.6124
Reranker score: -11.3137
Customer support is available Monday through Friday from 9 AM to 5 PM.
---


The ranking may change after reranking.

That's the point.

The first stage asks:

> **Which chunks look like reasonable candidates?**

The reranker asks:

> **Which candidate is most relevant to this exact query?**

# 10. Retrieval Score vs Reranker Score

Notice that we now have two different scores:

```text
retrieval_score
reranker_score
```

They are produced by different models for different purposes.

We should not assume that the numbers are directly comparable.

For example, this does **not** imply that:

```text
0.80 retrieval score
```

is somehow equivalent to:

```text
0.80 reranker score
```

The important thing is the ranking within each stage.

This is the same general lesson we saw with hybrid retrieval:

> **Different models can produce scores on different scales.**

# 11. The Two-Stage Retrieval Architecture

We can now visualize the process:

```text
                       Query
                         │
                         ▼
                First-Stage Retriever
                         │
                         ▼
                  Candidate Set
                         │
                         ▼
                   Cross-Encoder
                         │
                         ▼
                    Reranking
                         │
                         ▼
                  Top Candidates
                         │
                         ▼
                      Context
                         │
                         ▼
                        LLM
```

The first stage is optimized for finding candidates efficiently.

The second stage is optimized for making a more detailed relevance judgment.

# 12. Why We Don't Rerank the Entire Database

Suppose we have:

```text
1,000,000 documents
```

A cross-encoder would need to evaluate a very large number of query-document pairs if we applied it to the entire corpus.

Instead:

```text
1,000,000 documents
        ↓
Fast retrieval
        ↓
100 candidates
        ↓
Cross-encoder
        ↓
Top 10
```

The first-stage retriever dramatically reduces the amount of work the expensive model performs.

This is the core reason multi-stage retrieval works.

# 13. Choosing Candidate Count

The number of candidates passed to the reranker is an important parameter.

For example:

```text
top_k = 5
```

might be too small if the first-stage retriever occasionally misses the correct chunk.

But:

```text
top_k = 500
```

may create unnecessary reranking latency.

The trade-off is:

```text
More candidates
    ↓
Potentially higher recall
    ↓
More reranking work
    ↓
Higher latency

Fewer candidates
    ↓
Lower latency
    ↓
Greater risk of missing relevant evidence
```

There is no universal value of `k`.

It should be selected using the characteristics and evaluation results of the application.

# 14. Reranking Does Not Recover Missing Documents

This is one of the most important limitations.

Suppose the correct document is ranked:

```text
Position 101
```

but we retrieve:

```text
Top 20
```

The reranker never sees it.

Therefore:

```text
Retriever misses document
        ↓
Document not in candidates
        ↓
Reranker cannot recover it
```

Reranking improves the ordering of candidates.

It does not magically search the entire corpus.

This is why first-stage retrieval still needs strong recall.

# 15. Reranking and Hybrid Retrieval

Now combine what we learned in the previous tutorial.

We can have:

```text
                         Query
                           │
                 ┌─────────┴─────────┐
                 ↓                   ↓
          Dense Retrieval      Lexical Retrieval
                 ↓                   ↓
                 └─────────┬─────────┘
                           ↓
                          RRF
                           ↓
                    Candidate Set
                           ↓
                       Reranker
                           ↓
                    Final Ranking
                           ↓
                        Context
                           ↓
                          LLM
```

Each stage has a distinct job:

```text
Dense
→ semantic matching

Lexical
→ exact-term matching

RRF
→ combine retrieval rankings

Reranker
→ fine-grained relevance judgment
```

# 16. Reranking and Context Quality

The LLM does not see our entire knowledge base.

It sees the context we provide.

If we select:

```text
Top 5 poorly ranked chunks
```

the LLM receives poor evidence.

If reranking gives us:

```text
Top 5 highly relevant chunks
```

the LLM receives stronger evidence.

So:

```text
Better ranking
      ↓
Better evidence selection
      ↓
Better context
      ↓
Potentially better answer
```

Reranking can improve evidence selection.

It does **not** guarantee that the generated answer is correct.

# 17. Latency Trade-Off

The major cost of reranking is computation.

Without reranking:

```text
Query
  ↓
Vector Search
  ↓
Context
```

With reranking:

```text
Query
  ↓
Vector Search
  ↓
Candidate Set
  ↓
Cross-Encoder
  ↓
Context
```

We've added another model inference step.

For an interactive application, consider:

- Candidate count
- Reranker model size
- Hardware
- Batching
- Query volume
- Latency requirements

This is why production RAG is a systems problem, not just a model-selection problem.

# 18. Reranking Is a Precision Tool

A useful mental model is:

```text
Retriever
    ↓
Broad but useful
```

and:

```text
Reranker
    ↓
Narrow and precise
```

The retriever searches a large space.

The reranker makes a more expensive judgment over a much smaller space.

```text
Large search space
        ↓
Cheap approximation
        ↓
Small candidate space
        ↓
Expensive relevance judgment
```

# 19. Inspect the Ranking Changes

One of the best ways to understand reranking is to compare the rankings directly.

In [8]:
print("FIRST-STAGE RANKING")
print("=" * 40)

for rank, candidate in enumerate(candidates, start=1):
    print(f"{rank}. {candidate['id']}")

print("\nRERANKED")
print("=" * 40)

for rank, candidate in enumerate(
    reranked_candidates,
    start=1
):
    print(f"{rank}. {candidate['id']}")

FIRST-STAGE RANKING
1. refund-condition
2. refund-deadline
3. refund-processing
4. refund-method
5. support-hours

RERANKED
1. refund-condition
2. refund-deadline
3. refund-method
4. refund-processing
5. support-hours


Ask:

- Which documents moved up?
- Which moved down?
- Did the most directly relevant evidence move toward the top?
- Did loosely related chunks move down?

This is how we should think about reranking experimentally.

Don't assume that adding a reranker improves the system.

**Measure it.**

# 20. How We Evaluate a Reranker

Suppose we have a set of evaluation questions.

For each question, we know which document contains the relevant evidence.

We can compare:

```text
Retriever only
```

against:

```text
Retriever + reranker
```

Useful retrieval metrics include:

- Recall@k
- Precision@k
- MRR
- NDCG

For example:

> Did the relevant chunk appear in the top 5 before reranking?

and:

> Did it appear in the top 5 after reranking?

This turns reranking from a theoretical technique into an experimentally testable component.

# Key Takeaways

1. Retrieval and reranking solve different problems.
2. The first retrieval stage should find a broad candidate set.
3. Reranking determines which candidates are most relevant to the exact query.
4. Bi-encoders encode queries and documents independently, enabling efficient large-scale retrieval.
5. Cross-encoders examine the query and candidate together and can provide stronger relevance judgments.
6. We should not run an expensive cross-encoder over an entire million-document corpus for every query.
7. A common architecture is:

```text
Fast retrieval
    ↓
Candidate set
    ↓
Expensive reranking
    ↓
Final candidates
```

8. Reranking cannot recover a document that the first-stage retriever failed to retrieve.
9. Candidate count is a trade-off between recall and latency.
10. Reranker scores should not be directly compared with vector similarity scores.
11. Reranking can improve evidence selection, but it does not guarantee a correct generated answer.
12. The right way to judge a reranker is through retrieval evaluation, not intuition alone.

# What's Next?

Our retrieval pipeline is now considerably stronger:

```text
Query
  ↓
Dense Retrieval
  +
Lexical Retrieval
  ↓
RRF
  ↓
Candidate Set
  ↓
Reranking
  ↓
High-Quality Candidates
```

But one important problem remains.

Even if we retrieve the right chunks, **how should we construct the context that we give to the LLM?**

Should we simply concatenate every retrieved chunk?

What order should the chunks appear in?

Should duplicate information be removed?

How much context should we include?

What happens when chunks contain conflicting information?

These are **context engineering** problems.

In the next tutorial, we'll move from:

> **finding relevant information**

to:

> **constructing the best possible context for the LLM.**